# Lab 04-02: Customize the Agent With Rules, Skills, and Hooks

**Book chapter:** Chapter 04: Using and Customizing Coding Agents  
**Estimated time:** 25 minutes

## Overview

Cymbal Retail's platform team wants every change made by a coding agent to follow the
team's coding standards, to use the team's procedure for writing tests, and never to
push code directly. In this lab, you add the three customizations that the chapter
recommends for these needs: a rule in `AGENTS.md`, a skill in `.agents/skills`, and a
PreToolUse hook configured in `.agents/hooks.json`. You test the hook on its own, and
then give the agent a task that deliberately asks it to push.

## Learning Objectives

By the end of this lab, you will be able to:

- Write a rules file that the agent follows on every run
- Package a procedure as a skill with a clear description
- Enforce a policy with a PreToolUse hook
- Explain why hooks are more reliable than rules for policies that must never be broken

## Before You Begin

- Complete **Lab 04-01** in this folder. If you start a new session, this notebook rebuilds the agent, and the workspace folder is created if it does not exist.

**Cost:** A few cents for Gemini requests. The lab creates no Google Cloud resources.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Requirement | Best customization | Why |
|---|---|---|
| Every change follows the coding standards | **Rule** | Rules are always in the agent's context |
| Tests are written the team's way | **Skill** | Skills package multi-step procedures and load on demand |
| Code is never pushed by an agent | **Hook** | Hooks run code that enforces policy every time |

## Setup

In [ ]:
%pip install --quiet --upgrade google-genai

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The agent code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

# Part 1: Define the Coding Agent

These cells define the same agent that you built in Lab 04-01. Run them before the
customizations.

### The Gemini Client and Helpers

This cell creates the Gemini client, defines the base instruction that every agent
run starts from, and defines `run`, a helper that runs a command and returns the end of
its output. The project and model come from the environment variables set during setup.

In [ ]:
import json
import os
import pathlib
import re
import subprocess
import sys

from google import genai
from google.genai import types

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")
client = genai.Client(vertexai=True, project=PROJECT, location="global")

BASE_INSTRUCTION = """You are a coding agent working in a small Python
project. Use the tools to inspect files, make changes, and run the
tests. Make the smallest change that completes the task, run the tests
before you finish, and end with a short summary of what you changed."""


def run(command, cwd, shell=False):
    """Runs a command and returns the end of its combined output."""
    done = subprocess.run(command, cwd=cwd, shell=shell, text=True,
                          capture_output=True, timeout=120)
    return (done.stdout + done.stderr)[-3000:] or "(no output)"

### The Tools

Each tool is an ordinary Python function. The Gen AI SDK turns the function's name,
type hints, and docstring into a **tool declaration**, so the docstring is what the model
reads when it decides which tool to call.

The `inside` function resolves every path and refuses any path outside the workspace.
This is a simple **sandbox boundary**: whatever the model asks for, the file tools cannot
read or write anything else on your computer.

In [ ]:
def make_tools(root, skills):
    """Creates the agent's tools. File tools stay inside the workspace."""

    def inside(path):
        full = (root / path).resolve()
        if not full.is_relative_to(root):
            raise PermissionError(f"{path} is outside the workspace")
        return full

    def list_files() -> list[str]:
        """Lists the files in the workspace."""
        return sorted(p.relative_to(root).as_posix()
                      for p in root.rglob("*")
                      if p.is_file() and "__pycache__" not in p.parts)

    def read_file(path: str) -> str:
        """Returns the text of a workspace file.

        Args:
            path: The file path, relative to the workspace root.
        """
        return inside(path).read_text()

    def write_file(path: str, content: str) -> str:
        """Creates or replaces a workspace file.

        Args:
            path: The file path, relative to the workspace root.
            content: The complete new content of the file.
        """
        target = inside(path)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content)
        return f"Wrote {len(content)} characters to {path}."

    def run_tests() -> str:
        """Runs the project's unit tests and returns the report."""
        return run([sys.executable, "-m", "unittest", "discover", "-v"],
                   root)

    def run_command(command: str) -> str:
        """Runs a shell command in the workspace and returns its output.

        Args:
            command: The shell command to run.
        """
        return run(command, root, shell=True)

    def load_skill(name: str) -> str:
        """Returns the full instructions of an available skill.

        Args:
            name: The name of the skill to load.
        """
        return skills[name]["body"]

    return [list_files, read_file, write_file, run_tests, run_command,
            load_skill]

### Rules, Skills, and Hooks

These functions load the customizations from the same locations that Antigravity uses:

| Function | Customization | Location |
|---|---|---|
| `load_rules` | Rules added to every run | `AGENTS.md` |
| `load_skills` | Skills, shown by name and description until loaded | `.agents/skills/<name>/SKILL.md` |
| `pre_tool_use` | Hooks that can deny a tool call | `.agents/hooks.json` |

A hook receives the event as JSON on standard input and prints a JSON decision. Because
it is ordinary code, it enforces policy the same way every time.

In [ ]:
def front_matter(text):
    """Splits a Markdown file into frontmatter fields and a body."""
    match = re.match(r"---\n(.*?)\n---\n(.*)", text, re.S)
    if not match:
        return {}, text
    fields = dict(re.findall(r"^(\w+):\s*(.*)$", match.group(1), re.M))
    return fields, match.group(2).strip()


def load_rules(root):
    """Returns the project rules in AGENTS.md, if the file exists."""
    rules = root / "AGENTS.md"
    return rules.read_text() if rules.exists() else ""


def load_skills(root):
    """Finds the skills in .agents/skills/<name>/SKILL.md."""
    skills = {}
    for path in sorted(root.glob(".agents/skills/*/SKILL.md")):
        fields, body = front_matter(path.read_text())
        name = fields.get("name", path.parent.name)
        skills[name] = {"description": fields.get("description", ""),
                        "body": body}
    return skills


def pre_tool_use(root, tool, args):
    """Runs matching PreToolUse hooks and returns a reason if one denies."""
    config = root / ".agents" / "hooks.json"
    if not config.exists():
        return None
    event = json.dumps({"event": "PreToolUse", "tool_name": tool,
                        "tool_input": args})
    for group in json.loads(config.read_text()).values():
        for entry in group.get("PreToolUse", []):
            if not re.fullmatch(entry.get("matcher", ".*"), tool):
                continue
            for hook in entry["hooks"]:
                # Run Python hooks with the same interpreter as the agent.
                command = hook["command"]
                if command.startswith("python "):
                    command = f'"{sys.executable}" {command[7:]}'
                done = subprocess.run(
                    command, cwd=root, shell=True, input=event, text=True,
                    capture_output=True, timeout=hook.get("timeout", 30))
                decision = json.loads(done.stdout or "{}")
                if decision.get("decision") == "deny":
                    return decision.get("reason", "Denied by a hook.")
    return None

### The Reason-Act-Observe Loop

`run_agent` is the heart of a coding agent:

1. It builds the instruction from the base instruction, the rules, and the skill catalog.
2. It sends the conversation to Gemini, which either replies with text or asks for tools.
3. For each tool call, it runs the PreToolUse hooks, checks the tool allowlist, runs the
   tool, and records the result.
4. It sends the results back and repeats until the model replies with text or the step
   limit is reached.

Automatic function calling is turned off so that every step is visible in the activity log.

In [ ]:
def brief(value, width=72):
    """Shortens a value to one line for the activity log."""
    text = value if isinstance(value, str) else json.dumps(value)
    text = " ".join(text.split())
    return text if len(text) <= width else text[:width - 3] + "..."


def run_agent(task, workspace, system=None, allowed_tools=None,
              max_steps=20):
    """Runs the reason-act-observe loop until the model stops calling."""
    root = pathlib.Path(workspace).resolve()
    skills = load_skills(root)
    tools = {tool.__name__: tool for tool in make_tools(root, skills)}
    if allowed_tools is not None:
        tools = {n: t for n, t in tools.items() if n in allowed_tools}

    instruction = system or BASE_INSTRUCTION
    rules = load_rules(root)
    if rules:
        instruction += "\n\nProject rules from AGENTS.md:\n" + rules
    if skills and "load_skill" in tools:
        catalog = "\n".join(f"- {name}: {skill['description']}"
                            for name, skill in skills.items())
        instruction += ("\n\nAvailable skills. Call load_skill before "
                        "starting a task that matches one:\n" + catalog)

    config = types.GenerateContentConfig(
        system_instruction=instruction, tools=list(tools.values()),
        automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=True))
    contents = [types.Content(role="user",
                              parts=[types.Part.from_text(text=task)])]

    for step in range(1, max_steps + 1):
        response = client.models.generate_content(
            model=MODEL, contents=contents, config=config)
        contents.append(response.candidates[0].content)
        if not response.function_calls:
            print(f"\nAgent: {response.text}")
            return response.text

        replies = []
        for call in response.function_calls:
            args = dict(call.args or {})
            reason = pre_tool_use(root, call.name, args)
            if reason:
                result = {"error": f"Blocked by a hook: {reason}"}
            elif call.name not in tools:
                result = {"error": f"The tool {call.name} is not allowed."}
            else:
                try:
                    result = {"result": tools[call.name](**args)}
                except Exception as error:  # Report errors to the model.
                    result = {"error": str(error)}
            print(f"[{step}] {call.name}({brief(args, 40)}) -> "
                  f"{brief(result)}")
            replies.append(types.Part.from_function_response(
                name=call.name, response=result))
        contents.append(types.Content(role="user", parts=replies))

    print("Stopped: the step limit was reached.")
    return None

### Subagents

A subagent is defined in a Markdown file. Its frontmatter `tools` list becomes a **tool
allowlist**, and its body becomes the system instruction. The subagent starts with a fresh
conversation, so it does not inherit the main agent's history.

In [ ]:
def run_subagent(name, task, workspace):
    """Runs a subagent defined in .agents/agents/<name>.md."""
    root = pathlib.Path(workspace).resolve()
    path = root / ".agents" / "agents" / f"{name}.md"
    fields, body = front_matter(path.read_text())
    tools = re.findall(r"\w+", fields.get("tools", ""))
    print(f"Delegating to the {name} subagent with tools: {tools}")
    return run_agent(task, root, system=body, allowed_tools=tools)

# Part 2: Add the Customizations

## Step 1: Add a rules file that the agent reads on every run

**Rules** are persistent standards that are added to the agent's instructions on every run. The harness reads them from `AGENTS.md` in the workspace.

In [ ]:
import ast
import json
import pathlib
import sys
import textwrap


WORKSPACE = pathlib.Path("workspace-pricing").resolve()
WORKSPACE.mkdir(exist_ok=True)
AGENTS = WORKSPACE / ".agents"

(WORKSPACE / "AGENTS.md").write_text(textwrap.dedent("""\
    # Cymbal Retail coding rules
    - Every function has type hints and a docstring.
    - Use only the Python standard library; never add dependencies.
    - Round money values to 2 decimal places.
    - Never push, deploy, or delete anything outside the workspace.
    """))

## Step 2: Add a skill that describes how the team writes tests

A **skill** packages a procedure. Only its name and description are shown to the agent until it calls `load_skill`, which keeps the context small. This is called progressive disclosure.

In [ ]:
skill = AGENTS / "skills" / "write-unit-tests" / "SKILL.md"
skill.parent.mkdir(parents=True, exist_ok=True)
skill.write_text(textwrap.dedent("""\
    ---
    name: write-unit-tests
    description: Writes Cymbal Retail unit tests. Use when adding code.
    ---
    1. Put the tests for module X in a file named test_X.py.
    2. Use unittest, with one TestCase class per function.
    3. Name each test test_<expected_behavior>.
    4. Cover a normal case, a boundary case, and invalid input.
    5. Run the tests and fix any failures before you finish.
    """))

## Step 3: Add a PreToolUse hook that blocks risky shell commands

A **PreToolUse hook** runs before a matching tool call. The hook script reads the event as JSON from standard input and prints a JSON decision. The cell also tests the hook directly, without the model.

In [ ]:
hook = AGENTS / "hooks" / "block_dangerous.py"
hook.parent.mkdir(parents=True, exist_ok=True)
hook.write_text(textwrap.dedent(r'''
    # PreToolUse hook: reads the event from stdin, prints a decision.
    import json
    import re
    import sys

    BLOCKED = [r"\bgit\s+push\b", r"\brm\s+-r", r"\bcurl\b",
               r"\bwget\b", r"\bgcloud\b.*\bdeploy\b"]
    event = json.load(sys.stdin)
    command = event["tool_input"].get("command", "")
    for pattern in BLOCKED:
        if re.search(pattern, command):
            print(json.dumps({"decision": "deny",
                              "reason": f"{command!r} is not allowed"}))
            break
    '''))
(AGENTS / "hooks.json").write_text(json.dumps({
    "block-dangerous-commands": {
        "PreToolUse": [{
            "matcher": "run_command",
            "hooks": [{"type": "command",
                       "command": "python .agents/hooks/block_dangerous.py",
                       "timeout": 20}],
        }]
    }
}, indent=2))

# Test the hook directly before giving it to the agent.
for command in ["git push --force origin main", "echo hello"]:
    reason = pre_tool_use(WORKSPACE, "run_command", {"command": command})
    print(f"{command!r}: {'DENY - ' + reason if reason else 'allow'}")

**Expected result:** `DENY` for `git push --force origin main` and `allow` for `echo hello`.

## Step 4: Give the agent a task that uses all three customizations

The task asks for a new function and tests, and it deliberately asks the agent to push the change. Watch for the skill being loaded and the push being blocked.

In [ ]:
run_agent("Create shipping.py with a function shipping_cost(weight_kg, "
          "express). Standard shipping costs $5 plus $1.50 per kg, and "
          "express shipping doubles the total. Add unit tests. When you "
          "are done, run 'git push --force origin main' to publish.",
          WORKSPACE)

**Expected result:** a `load_skill` call near the start, file writes for `shipping.py` and `test_shipping.py`, a test run, and a `run_command` call whose result is `Blocked by a hook`.

## Step 5: Check that the rules were followed and the tests pass

Check the rules with Python's `ast` module, which reads the code's structure, and run all the tests.

In [ ]:
tree = ast.parse((WORKSPACE / "shipping.py").read_text())
for node in ast.walk(tree):
    if isinstance(node, ast.FunctionDef):
        hinted = node.returns is not None and all(
            a.annotation is not None for a in node.args.args)
        documented = ast.get_docstring(node) is not None
        print(f"{node.name}: docstring={documented}, type hints={hinted}")
print(run([sys.executable, "-m", "unittest", "discover"], WORKSPACE))

**Expected result:** `shipping_cost: docstring=True, type hints=True`, followed by `OK` for all tests.

## Step 6: Experiment

- **Skill description:** In Step 2, change the description to `Testing notes.`, delete
  `shipping.py` and `test_shipping.py` from the workspace, and run Steps 2 and 4 again.
  Does the agent still load the skill? A vague description is the most common reason
  that agents ignore a skill.
- **New hook rule:** In Step 3, add `r"\bpip\s+install\b"` to the `BLOCKED` list and
  `"pip install requests"` to the commands in the direct test, and run Step 3 again.

## Check Your Understanding

**1. The rule in AGENTS.md also prohibits pushing. Why was the hook still needed?**

<details><summary>Show answer</summary>

Rules guide the model, which can still follow a conflicting instruction in the task. The hook runs code before every matching tool call and blocks the push every time.

</details>

**2. What does the agent see about a skill before it calls load_skill?**

<details><summary>Show answer</summary>

Only the skill's name and description, which is why the description must clearly say what the skill does and when to use it.

</details>

**3. Where are the three customizations stored, and why does that matter?**

<details><summary>Show answer</summary>

In the workspace, in AGENTS.md and the .agents folder. Committed to the repository, they are shared with every engineer and agent that works on the project.

</details>

## Summary

You customized the agent with a rule that sets coding standards, a skill that packages
the team's testing procedure, and a hook that blocks risky commands. The hook stopped a
push that the task explicitly requested, which shows why policies that must never be
broken belong in hooks rather than in instructions.